In [1]:
#| include: false
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns

from utils.style import COLORS, CYCLE, apply_style

apply_style()
np.set_printoptions(precision=4, suppress=True, linewidth=110)
_ = pl.Config.set_float_precision(4)

We use the Madrid listings introduced in
[the first part of the topic](01-stats-population-samples.ipynb#our-data-dwellings-for-sale-in-madrid).
The file has 31 columns (described in `data/processed/idealista/README.md`); in this
course we mostly use the following fourteen:

In [2]:
listings = pl.read_csv("data/processed/idealista/madrid_listings_2018.csv").select(
    "listing_id", "district", "zone", "condition",
    "price", "price_m2", "area_m2", "bedrooms", "bathrooms",
    "floor", "lift", "terrace", "construction_year", "dist_centre_km",
)

- `listing_id`: identifier of the listing on the portal.
- `district`: district of Madrid where the dwelling is (the city has 21).
- `zone`: `Centre` (the seven central districts, roughly inside the M-30 ring road)
  or `Periphery` (the other fourteen).
- `condition`: state of the dwelling: `To renovate`, `Good` (second-hand in good
  condition) or `New` (newly built).
- `price`: asking price, in euros.
- `price_m2`: asking price per square metre of floor area, in euros.
- `area_m2`: floor area of the dwelling, in m².
- `bedrooms` and `bathrooms`: number of bedrooms (0 for a studio) and of bathrooms.
- `floor`: floor of the dwelling (−1 basement, 0 ground floor, 1 first floor…).
  Missing for a few listings.
- `lift` and `terrace`: `true` if the building has a lift and if the dwelling has a
  terrace.
- `construction_year`: year in which the building was built.
- `dist_centre_km`: distance to the city centre (Puerta del Sol), in km.

## Variables and data

::: {.callout-note}
## Definition

- A **statistical variable** (or simply a variable) is an observable characteristic of
  the individuals of a population that can take different **values**, or
  **categories**, from one individual to another. We denote variables by upper-case
  letters: $X$, $Y$, $X_j$.
- An **observation** is the value of a variable for one particular individual. We write
  $x_i$ for the value of the variable $X$ for individual $i$.
- An **observed sample** (the **data**) is the list $(x_1, \dots, x_n)$ of the
  observations of a variable on the $n$ individuals observed. It is a list, not a set:
  two individuals can have the same value, and both count.
- With $p$ variables $X_1, \dots, X_p$, the data form the **data matrix**
  $\mathbf{X} = (x_{ij})$ of size $n \times p$, where $x_{ij}$ is the value of variable
  $X_j$ for individual $i$.
:::

The word *sample* has two related uses. In
[part 1](01-stats-population-samples.ipynb#census-and-sample) the sample is the set of
**individuals** selected; the observed sample is the list of **values** of a variable for
those individuals.

::: {.callout-tip}
## Looking ahead: the random version of these concepts

In descriptive statistics nothing is random: each dwelling has its price, and the data
are fixed numbers. Probability adds chance. If an individual is **drawn at random**, the
value of a variable for the individual that will be drawn is a *random variable*, written
in upper case; if $n$ individuals are drawn, their values form a *random sample*
$(X_1, \dots, X_n)$, and the observed sample $(x_1, \dots, x_n)$ is one *realisation* of it.
The *Probability* course (Topic 2) defines these concepts and proves that the distribution
of a randomly drawn observation is the distribution of the variable in the population.

| Here (fixed) | In probability (random) |
|---|---|
| Variable $X$ and its population distribution | Random variable $X$, with the same distribution |
| Observation $x_i$ | Random observation $X_i$ |
| Observed sample $(x_1, \dots, x_n)$ | Random sample $(X_1, \dots, X_n)$ |
| Statistic computed on the data, $\bar x$ | Statistic as a random variable, $\bar X$ |
:::

In the data matrix **each row is an individual and each column a variable**. This is
exactly the structure of a Polars `DataFrame`, with $n = 75{,}735$ rows and $p = 14$
columns here:

In [3]:
listings.head()

listing_id,district,zone,condition,price,price_m2,area_m2,bedrooms,bathrooms,floor,lift,terrace,construction_year,dist_centre_km
A10000037964896093228,Hortaleza,Periphery,Good,"255,000.0","2,628.9",97,3,2,4,True,False,1966,7.829
A10000072440601830803,Puente de Vallecas,Periphery,To renovate,"82,000.0","1,322.6",62,2,1,1,True,False,1965,6.435
A10000538600815177437,Puente de Vallecas,Periphery,Good,"133,000.0","1,985.1",67,3,1,3,False,True,1976,4.836
A10000654405436195291,Ciudad Lineal,Periphery,Good,"204,000.0","1,133.3",180,3,2,0,True,False,1976,4.665
A10000872160480475600,Carabanchel,Periphery,Good,"161,000.0","2,981.5",54,2,1,1,False,False,1955,4.063


## Classification of variables

The type of a variable decides which tables, graphs and summaries make sense.

**Qualitative (categorical) variables**: their values are qualities, not
quantities.

- **Nominal**: the categories have **no** natural order. For example, the district
  or the zone.
- **Ordinal**: the categories have a natural order, but the distance between them is
  not defined. For example, the condition of the dwelling: "to renovate" is worse
  than "good", which is worse than "new", but it makes no sense to ask *how much*
  worse.
- **Binary** (dichotomous) variables have only two categories, such as having a lift
  or not. They are usually coded as 1 (yes) and 0 (no), and then their mean is the
  proportion of ones.

**Quantitative variables**: their values are numbers that express an amount,
measured in some units.

- **Discrete**: they take separate values, usually whole numbers, that can be
  counted. For example, the number of bedrooms.
- **Continuous**: they can take, at least in theory, any value in an interval. For
  example, the price, the floor area or the distance to the centre.

```{mermaid}
%%| echo: false
%%| label: fig-types
%%| fig-cap: "Classification of statistical variables, with examples from the Madrid listings."
flowchart TD
    V["Variable"] --> CL["Qualitative"]
    V --> CU["Quantitative"]
    CL --> N["Nominal<br/><i>district</i>"]
    CL --> O["Ordinal<br/><i>condition</i>"]
    CU --> D["Discrete<br/><i>bedrooms</i>"]
    CU --> C["Continuous<br/><i>price, area</i>"]
```

::: {.callout-warning}
## Being recorded as a number does not make a variable quantitative

Many qualitative variables are recorded with numbers: postcodes, district codes
(01 for Centro, 04 for Salamanca…) or identifiers. Computing the mean of these codes
makes no sense. The type of a variable depends on **what it measures**, not on how
it is stored.
:::

::: {.callout-tip}
## Discrete or continuous in practice

Every measurement has limited precision: prices are rounded to the euro and areas to
the square metre. Strictly, all observed data are discrete. In practice we treat a
variable as **continuous** when it takes many different values and intermediate
values make sense. The price is continuous for practical purposes; the number of
bedrooms is not.
:::

## Measurement scales

A finer classification, due to Stevens [-@stevens1946], looks at the **operations**
that make sense with the values of a variable:

| Scale | What it allows | Admissible transformations | Example | Measures of centre |
|---|---|---|---|---|
| Nominal | Equal or different: $=$, $\neq$ | Any one-to-one relabelling | District | Mode |
| Ordinal | Also order: $<$, $>$ | Strictly increasing $g$ | Condition | Mode, median |
| Interval | Also differences: $x_i - x_j$ | $a + bx$, $b > 0$ | Year of construction, floor | Mode, median, mean |
| Ratio | Also ratios: $x_i / x_j$ | $bx$, $b > 0$ | Price, area, distance | All |

: Measurement scales, the operations they allow and the transformations that leave their meaning unchanged. {#tbl-scales}

The third column gives a precise criterion. A scale is defined by the transformations
that do not change the information it carries: the condition of a dwelling could be
coded 1, 2, 3 or 0, 5, 100, since any increasing coding keeps the order; temperatures
can be expressed in °C or °F, $F = 32 + 1.8\,C$; prices in euros or in dollars. A
statement about the data is **meaningful** for a scale if its truth does not depend on
which admissible transformation is used. For example, "the median condition in the
centre is higher than in the periphery" is meaningful for ordinal data, because
increasing transformations preserve medians, but a comparison of mean codes is not
([Topic 2](../02-univariate/02-centre-position.ipynb#measures-of-position-quantiles)).

The difference between the last two scales is the meaning of **zero**. On a ratio
scale zero means absence of the quantity (0 m², 0 €), so it makes sense to say that a
flat of 120 m² is *twice as large* as one of 60 m². On an interval scale zero is a
convention. A building from 2000 is not "twice as old" as one from 1000, and the
ground floor is numbered 0 only by custom (in other countries it is the first
floor). Differences, however, do make sense: a building from 1990 is 30 years newer
than one from 1960. Most quantitative variables in economics are on a ratio scale,
which allows percentages of change and relative measures such as the coefficient of
variation
([Topic 2](../02-univariate/03-dispersion-shape.ipynb#the-coefficient-of-variation)).

::: {.callout-note}
## Historical note: S. S. Stevens and the scales of measurement

The psychologist Stanley Smith Stevens proposed the four scales in a short paper in
*Science* in 1946 [@stevens1946], after a long debate on whether sensations such as
loudness could be "measured" at all. His classification is still the standard one,
although it has been criticised as too rigid: for example, answers on a scale from 1
to 5 are ordinal, but they are often averaged as if they were interval data.
:::

## Types of data by structure

How the data are organised in time also matters:

- **Cross-sectional data**: many individuals observed at the same time. For example,
  the dwellings for sale in Madrid in 2018, our data.
- **Time series**: a *single* individual or aggregate observed at many moments. For
  example, the INE housing price index, one value per quarter.
- **Panel (longitudinal) data**: *many* individuals, the *same* ones, observed at
  several moments. For example, the asking price of the same dwellings in each
  quarter they stay on the market: we can see whether the seller lowers the price.

| Type | Individuals | Moments | Example |
|---|---|---|---|
| Cross-section | many | one | Listings in Madrid in 2018 |
| Time series | one | many | Quarterly housing price index |
| Panel | many, the same ones | many | Price of each listing, quarter by quarter |

This course works with cross-sectional data; time series are the subject of a course
of their own.

## Types of variables in Python

Polars gives each column a **data type** (*dtype*), which says how it is stored. The
storage type does not necessarily match the statistical type:

In [4]:
pl.DataFrame({
    "variable": listings.columns,
    "Polars type": [str(t) for t in listings.dtypes],
    "statistical type": [
        "Identifier (not a study variable)",  # listing_id
        "Qualitative, nominal",               # district
        "Qualitative, nominal (binary)",      # zone
        "Qualitative, ordinal",               # condition
        "Quantitative, continuous",           # price
        "Quantitative, continuous",           # price_m2
        "Quantitative, continuous",           # area_m2
        "Quantitative, discrete",             # bedrooms
        "Quantitative, discrete",             # bathrooms
        "Quantitative, discrete (interval)",  # floor
        "Qualitative, binary",                # lift
        "Qualitative, binary",                # terrace
        "Quantitative, discrete (interval)",  # construction_year
        "Quantitative, continuous",           # dist_centre_km
    ],
})

variable,Polars type,statistical type
listing_id,String,Identifier (not a study variable)
district,String,"Qualitative, nominal"
zone,String,"Qualitative, nominal (binary)"
condition,String,"Qualitative, ordinal"
price,Float64,"Quantitative, continuous"
price_m2,Float64,"Quantitative, continuous"
area_m2,Int64,"Quantitative, continuous"
bedrooms,Int64,"Quantitative, discrete"
bathrooms,Int64,"Quantitative, discrete"
floor,Int64,"Quantitative, discrete (interval)"


Two remarks:

- `construction_year` is stored as an integer, and its mean can be computed. It
  makes sense (the "average age" of the buildings), because differences between
  years are meaningful.
- `condition` is stored as text (`String`), so Polars does not know that its
  categories are ordered. If we sort them, the order is alphabetical, which is not
  the logical one:

In [5]:
listings["condition"].unique().sort()

condition
Good
New
To renovate


To declare an ordinal variable we use the type `pl.Enum`, which keeps the categories
**in the order given**. Sorting and comparisons then follow the logical order:

In [6]:
condition_order = pl.Enum(["To renovate", "Good", "New"])
condition = listings["condition"].cast(condition_order)

condition.unique().sort()

condition
To renovate
Good
New


In [7]:
# Proportion of listings in good condition or better
round((condition >= "Good").mean(), 3)

0.813

The last expression uses two ideas of this topic: the comparison $\geq$ only makes
sense because the variable is **ordinal**, and the mean of a **binary** variable
(true = 1, false = 0) is a **proportion**.

## Summary of the topic

- Statistics studies how to draw conclusions from data that show **variability**,
  measuring the **uncertainty** of those conclusions.
- **Descriptive statistics** summarises the observed data. **Inference** generalises
  from the sample to the population, using **probability**.
- The **population** ($N$) is the set of interest; the **sample** ($n$) is the part
  observed. A representative sample must be chosen **at random**, and a larger sample
  does not correct a selection bias.
- A **parameter** describes the population, and is fixed and unknown. A
  **statistic** is computed from a sample and changes from sample to sample.
- Variables are **qualitative** (nominal or ordinal) or **quantitative** (discrete or
  continuous). The **measurement scale** decides which operations and summaries make
  sense.

## Exercises

::: {#exr-classify}
## Classifying variables

Classify the following variables of a dwelling by type (qualitative nominal or
ordinal, quantitative discrete or continuous) and measurement scale:

a) Postcode.
b) Energy rating (A, B, C, …, G).
c) Number of bedrooms.
d) Monthly rent (€).
e) Type of heating (none, individual, central).
f) Distance to the nearest metro station (km).
g) Year of the last renovation.
h) Owner's satisfaction with the neighbourhood, on a scale from 1 to 5.
:::

::: {.callout-tip collapse="true"}
## Solution

a) Qualitative nominal, although it is written with numbers.

b) Qualitative ordinal: A is better than B, but the "distance" between ratings is
not defined.

c) Quantitative discrete, ratio scale (a flat with 4 bedrooms has twice as many as
one with 2).

d) Quantitative continuous, ratio scale.

e) Qualitative nominal.

f) Quantitative continuous, ratio scale.

g) Quantitative discrete, interval scale: subtracting years makes sense (years since
the renovation), dividing them does not.

h) Qualitative ordinal. Computing its mean is common, but it assumes that the
distance between 1 and 2 is the same as between 4 and 5.
:::

::: {#exr-types-python}
## Types in Python

a) Compute, with Polars, the proportion of listings with a terrace.
b) Polars computes `listings["listing_id"]` without complaining if we sort it.
   Would it make sense to compute its median? Why?
c) Declare `zone` as an `Enum`. Does the order of its categories matter?
:::

::: {.callout-tip collapse="true"}
## Solution

In [8]:
print(f"a) Proportion with terrace: {listings['terrace'].mean():.3f}")
zone = listings["zone"].cast(pl.Enum(["Centre", "Periphery"]))
print("c)", zone.unique().sort().to_list())

a) Proportion with terrace: 0.351
c) ['Centre', 'Periphery']


a) About a third of the listings have a terrace. The mean of a binary variable is a
proportion.

b) No. The identifier only labels each listing; it is a qualitative variable, even
if it can be sorted as text.

c) The order of an `Enum` only matters for ordinal variables. `zone` is nominal, so
any order is valid; we only choose one for tables and graphs.
:::

## References {.unnumbered}

::: {#refs}
:::